In [4]:
%pip install pandas numpy
import pandas as pd
import numpy as np
 
#load industrial machine sensor data from local data
df = pd.read_csv('data/ai4i2020.csv')

#force notebook to display first 5 rows
df.head()


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [5]:
# Check for any missing values across all columns and print summary information
print("--- Missing Values Check ---")
print(df.isnull().sum())

print("\n--- Structural Information ---")
print(df.info())

# Drop the unique IDs/serial numbers because they carry no physical engineering meaning
df_cleaned = df.drop(['UDI', 'Product ID'], axis=1)

# Check the new shape of our data (Rows, Columns)
print("\nNew shape after dropping metadata:", df_cleaned.shape)

--- Missing Values Check ---
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

--- Structural Information ---
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null 

In [6]:
# Count exactly how many normal operations (0) vs machine failures (1) exist
failure_counts = df_cleaned['Machine failure'].value_counts()
print("--- Class Distribution ---")
print(failure_counts)

# Calculate the percentage of failures
failure_percent = (failure_counts[1] / len(df_cleaned)) * 100
print(f"\nPercentage of actual machine failures: {failure_percent:.2%}")

--- Class Distribution ---
Machine failure
0    9661
1     339
Name: count, dtype: int64

Percentage of actual machine failures: 339.00%


In [7]:
# Convert the categorical 'Type' column into distinct numerical columns
df_encoded = pd.get_dummies(df_cleaned, columns=['Type'], drop_first=True)

# Let's inspect our new columns to see how 'Type' was transformed
print("--- New Columns After One-Hot Encoding ---")
print(df_encoded.columns.tolist())

# Preview the first 3 rows of our newly engineered dataset
df_encoded.head(3)

--- New Columns After One-Hot Encoding ---
['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Type_L', 'Type_M']


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,False,True
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,True,False
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,True,False


In [9]:
# Force-install scikit-learn directly into the notebook kernel context
%pip install scikit-learn

from sklearn.model_selection import train_test_split

# 1. Isolate the inputs (X). Drop the target and the cheating failure-mode columns
X = df_encoded.drop(['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], axis=1)

# 2. Isolate the target variable (y)
y = df_encoded['Machine failure']

# 3. Perform a Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("--- Data Split Verification ---")
print(f"Training feature matrix shape: {X_train.shape}")
print(f"Testing feature matrix shape: {X_test.shape}")
print(f"Actual failure counts in training data:\n{y_train.value_counts()}")

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 2.1 MB/s eta 0:00:04
   ----- ---------------------------------- 1.0/8.2 MB 1.8 MB/s eta 0:00:04
   ------- -------------------------------- 1.6/8.2 MB 2.0 MB/s eta 0:00:04
   ---------- ----------------------------- 2.1/8.2 MB 2.1 MB/s eta 0:00:03
   -------------- ------------------------- 2.9/8.2 MB 2.4 MB/s eta 0:00:03
   ----------------- ---------------------- 3.7/8.2 MB 2.5 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.2 MB 2.6 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.2 MB 2.6 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.2 MB 2.1 MB/s eta 0:00:02
   ---------------------- ----------------- 4.7/8.2 MB 2.1 MB/s eta 0:00:02
   ---------------------


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


--- Data Split Verification ---
Training feature matrix shape: (8000, 7)
Testing feature matrix shape: (2000, 7)
Actual failure counts in training data:
Machine failure
0    7729
1     271
Name: count, dtype: int64


In [10]:
# Force-install scikit-learn directly into the notebook kernel context
%pip install scikit-learn

from sklearn.model_selection import train_test_split

# 1. Isolate the inputs (X). Drop the target and the cheating failure-mode columns
X = df_encoded.drop(['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'], axis=1)

# 2. Isolate the target variable (y)
y = df_encoded['Machine failure']

# 3. Perform a Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("--- Data Split Verification ---")
print(f"Training feature matrix shape: {X_train.shape}")
print(f"Testing feature matrix shape: {X_test.shape}")
print(f"Actual failure counts in training data:\n{y_train.value_counts()}")

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
--- Data Split Verification ---
Training feature matrix shape: (8000, 7)
Testing feature matrix shape: (2000, 7)
Actual failure counts in training data:
Machine failure
0    7729
1     271
Name: count, dtype: int64



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler math object
scaler = StandardScaler()

# Fit the scaler on training data ONLY, and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Transform the test data using those exact same training calculations
X_test_scaled = scaler.transform(X_test)

# Convert back to a DataFrame for clean visual inspection
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)

print("--- Scaled Sample Comparison ---")
print("First row of raw inputs:\n", X_train.iloc[0])
print("\nFirst row of mathematically scaled inputs:\n", X_train_scaled_df.iloc[0])

--- Scaled Sample Comparison ---
First row of raw inputs:
 Air temperature [K]        302.0
Process temperature [K]    310.9
Rotational speed [rpm]      1456
Torque [Nm]                 47.2
Tool wear [min]               54
Type_L                     False
Type_M                      True
Name: 4058, dtype: object

First row of mathematically scaled inputs:
 Air temperature [K]        0.998914
Process temperature [K]    0.604282
Rotational speed [rpm]    -0.460607
Torque [Nm]                0.718305
Tool wear [min]           -0.843997
Type_L                    -1.234366
Type_M                     1.536207
Name: 0, dtype: float64


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# 1. Initialize the model with a balanced class weight to counter the 3.39% imbalance trap
baseline_model = LogisticRegression(class_weight='balanced', random_state=42)

# 2. Train the model using our scaled training data
baseline_model.fit(X_train_scaled, y_train)

# 3. Predict the status of the machines in our unseen test set
y_pred = baseline_model.predict(X_test_scaled)

print("--- Baseline Model Evaluation Report ---")
print(classification_report(y_test, y_pred))

--- Baseline Model Evaluation Report ---
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.82      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.82      0.57      2000
weighted avg       0.96      0.82      0.88      2000



In [13]:
from sklearn.ensemble import RandomForestClassifier

# 1. Initialize the Advanced Random Forest Model
advanced_model = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100)

# 2. Train the advanced model
advanced_model.fit(X_train_scaled, y_train)

# 3. Predict on the test set
y_pred_rf = advanced_model.predict(X_test_scaled)

print("--- Advanced Random Forest Evaluation Report ---")
print(classification_report(y_test, y_pred_rf))

--- Advanced Random Forest Evaluation Report ---
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1932
           1       0.73      0.69      0.71        68

    accuracy                           0.98      2000
   macro avg       0.86      0.84      0.85      2000
weighted avg       0.98      0.98      0.98      2000



In [14]:
# Extract the mathematical feature importance scores from our trained Random Forest
importances = advanced_model.feature_importances_

# Pair the scores with the column names and sort them from highest to lowest
feature_importance_df = pd.DataFrame({
    'Sensor Metric': X.columns,
    'Importance Score': importances
}).sort_values(by='Importance Score', ascending=False)

print("--- Core Engineering Feature Importances ---")
print(feature_importance_df.to_string(index=False))

--- Core Engineering Feature Importances ---
          Sensor Metric  Importance Score
            Torque [Nm]          0.324785
 Rotational speed [rpm]          0.283552
        Tool wear [min]          0.214098
    Air temperature [K]          0.097304
Process temperature [K]          0.065023
                 Type_L          0.008782
                 Type_M          0.006457
